## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

In [ ]:
df = pd.read_csv("/content/Ecommerce.csv")

df.head()

## 3. Dataset Information

In [ ]:
print("Dataset Information:")
df.info()

print("\n" + "="*50)

print("Column Names:")
print(df.columns.tolist())

print("\n" + "="*50)

print("Missing Values:")
print(df.isnull().sum())

print("\n" + "="*50)

print("Duplicate Rows:", df.duplicated().sum())

## 4. Statistical Summary

In [ ]:
print("Statistical Summary:")
df.describe()

## 5. Data Types of Features

In [ ]:
print("Data Types:")
print(df.dtypes)

## 6. Unique Values in Each Column

In [ ]:
print("Unique Values:")
print(df.nunique())

## 7. Check Column Names

In [ ]:
print("Column Names:")
print(df.columns.tolist())

## 8. Customer Distribution by Device Type

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(data=df, x="device_type")

plt.title("Customer Distribution by Device Type")
plt.xlabel("Device Type")
plt.ylabel("Number of Customers")

plt.show()

## 9. Customer Distribution by User Type

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(data=df, x="user_type")

plt.title("Customer Distribution by User Type")
plt.xlabel("User Type")
plt.ylabel("Number of Customers")

plt.show()

## 10. Revenue Distribution

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(data=df, x="revenue", bins=30, kde=True)

plt.title("Revenue Distribution")
plt.xlabel("Revenue")
plt.ylabel("Number of Transactions")

plt.show()

## 11. Revenue by Product Category

In [ ]:
category_revenue = (
    df.groupby("product_category")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=category_revenue.values,
    y=category_revenue.index
)

plt.title("Revenue by Product Category")
plt.xlabel("Total Revenue")
plt.ylabel("Product Category")

plt.show()

## 12. Revenue by Marketing Channel

In [ ]:
channel_revenue = (
    df.groupby("marketing_channel")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=channel_revenue.values,
    y=channel_revenue.index
)

plt.title("Revenue by Marketing Channel")
plt.xlabel("Total Revenue")
plt.ylabel("Marketing Channel")

plt.show()

## 13. Purchase Conversion Analysis

In [ ]:
purchase_counts = df["purchased"].value_counts()

plt.figure(figsize=(7, 5))

sns.barplot(
    x=purchase_counts.index,
    y=purchase_counts.values
)

plt.title("Purchase Conversion")
plt.xlabel("Purchased")
plt.ylabel("Number of Sessions")

plt.show()

## 14. Cart Abandonment Analysis

In [ ]:
cart_counts = df["cart_abandoned"].value_counts()

plt.figure(figsize=(7, 5))

sns.barplot(
    x=cart_counts.index,
    y=cart_counts.values
)

plt.title("Cart Abandonment Analysis")
plt.xlabel("Cart Abandoned")
plt.ylabel("Number of Sessions")

plt.show()

## 15. Customer Spending Analysis

In [ ]:
customer_spending = (
    df.groupby("customer_id")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("Top 10 Customers by Total Revenue:")
print(customer_spending.head(10))

## 16. Purchase Quantity Analysis

In [ ]:
quantity_by_category = (
    df.groupby("product_category")["quantity"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=quantity_by_category.values,
    y=quantity_by_category.index
)

plt.title("Quantity Sold by Product Category")
plt.xlabel("Total Quantity")
plt.ylabel("Product Category")

plt.show()

In [ ]:
## 17. Correlation Analysis

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(12, 8))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

## 18. Customer-Level Aggregation

In [ ]:
customer_data = df.groupby("customer_id").agg(
    total_revenue=("revenue", "sum"),
    total_quantity=("quantity", "sum"),
    total_sessions=("session_id", "nunique"),
    avg_pages_viewed=("pages_viewed", "mean"),
    avg_time_on_site=("time_on_site_sec", "mean"),
    total_purchases=("purchased", "sum"),
    total_cart_additions=("added_to_cart", "sum"),
    total_cart_abandonments=("cart_abandoned", "sum"),
    avg_discount=("discount_percent", "mean")
).reset_index()

customer_data.head()

## 19. Create Customer Behavior Features

In [ ]:
customer_data["purchase_rate"] = (
    customer_data["total_purchases"] /
    customer_data["total_sessions"]
)

customer_data["cart_abandonment_rate"] = (
    customer_data["total_cart_abandonments"] /
    customer_data["total_sessions"]
)

customer_data.head()

## 20. Select Features for Customer Segmentation

In [ ]:
segmentation_features = [
    "total_revenue",
    "total_quantity",
    "total_sessions",
    "avg_pages_viewed",
    "avg_time_on_site",
    "total_purchases",
    "total_cart_additions",
    "total_cart_abandonments",
    "avg_discount",
    "purchase_rate",
    "cart_abandonment_rate"
]

X = customer_data[segmentation_features]

X.head()

## 21. Check Missing Values in Segmentation Data

In [ ]:
print("Missing Values:")
print(X.isnull().sum())

## 22. Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

## 23. Elbow Method

In [ ]:
from sklearn.cluster import KMeans

inertia = []

K = range(2, 9)

for k in K:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))

plt.plot(K, inertia, marker="o")

plt.title("Elbow Method for Optimal Number of Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")

plt.show()

## 24. Silhouette Score Analysis

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []

for k in range(2, 9):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

    print(f"K = {k}, Silhouette Score = {score:.3f}")

## 25. Apply K-Means Clustering

In [ ]:
optimal_k = 2

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

customer_data["Cluster"] = kmeans.fit_predict(X_scaled)

customer_data.head()

## 26. Customer Segment Profile

In [ ]:
segment_profile = (
    customer_data.groupby("Cluster")[segmentation_features]
    .mean()
    .round(2)
)

segment_profile

## 27. Customer Distribution by Segment

In [ ]:
segment_counts = customer_data["Cluster"].value_counts().sort_index()

plt.figure(figsize=(8, 5))

sns.barplot(
    x=segment_counts.index,
    y=segment_counts.values
)

plt.title("Customer Distribution by Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")

plt.show()

## 28. Customer Segmentation Visualization

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=customer_data,
    x="total_revenue",
    y="purchase_rate",
    hue="Cluster",
    s=80
)

plt.title("Customer Segmentation: Revenue vs Purchase Rate")
plt.xlabel("Total Revenue")
plt.ylabel("Purchase Rate")

plt.show()

## 29. Business Recommendations

In [ ]:
print("Business Recommendations:")
print()
print("1. Focus on high-value customers in Cluster 0 with personalized offers.")
print("2. Encourage Cluster 1 customers with targeted discounts and product recommendations.")
print("3. Use customer purchase behavior to create personalized marketing campaigns.")
print("4. Reduce cart abandonment by improving the checkout experience.")
print("5. Monitor customer spending and purchase rates regularly to identify valuable customers.")

## 30. Conclusion

In [ ]:
print("Conclusion:")
print()
print("The E-Commerce Customer Segmentation Analysis successfully")
print("identified different customer groups based on their purchasing")
print("and engagement behavior. Customer-level features such as revenue,")
print("quantity, sessions, purchases, cart activity, and purchase rate")
print("were used for segmentation.")
print()
print("Using the Elbow Method and Silhouette Score, K-Means clustering")
print("was applied with 2 customer segments. Cluster 0 represents")
print("high-value and more active customers, while Cluster 1 represents")
print("lower-value customers with comparatively lower purchase activity.")
print()
print("These insights can help the business create personalized")
print("marketing campaigns, improve customer retention, offer targeted")
print("discounts, and increase overall customer revenue.")